# Data Source 04 · 공공정책

| 출처 | 접근 | 검증 상태 |
|---|---|---|
| California Prop 99 (`synthdid` 저장소 `california_prop99.csv`) | 무키 (GitHub raw) | A(열 구성) / B(라이선스) |
| World Bank API v2 | 무키 | A |
| NYC 311 Socrata (`erm2-nwe9`) | 무키(제한) | A |
| KOSIS / 공공데이터포털 | 무료키 | C |

Prop 99는 **구분자가 `;`** 이고 열이 State, Year, PacksPerCapita, treated이다. 합성대조(SCM)·DID 실습(Ch.7)의 대표 자료로 알려져 있다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def parse_prop99(raw):
    text = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else raw
    df = pd.read_csv(io.StringIO(text), sep=";")
    cols = {c.strip().lower(): c for c in df.columns}
    for need in ("state", "year", "packspercapita", "treated"):
        if need not in cols:
            raise ValueError("예상 열 없음: " + need + " / 실제: " + ", ".join(df.columns))
    df = df.rename(columns={cols["state"]: "State", cols["year"]: "Year",
                            cols["packspercapita"]: "PacksPerCapita", cols["treated"]: "treated"})
    return df

In [ ]:
URL = "https://raw.githubusercontent.com/synth-inference/synthdid/master/data/california_prop99.csv"
try:
    df = parse_prop99(fetch_bytes(URL))
    DATA_KIND = "REAL: California Prop 99 (synthdid repo)"
except Exception as e:
    print("온라인 수집 실패 -> 합성 패널 대체:", type(e).__name__, e)
    rng = np.random.default_rng(2026041)
    rows = []
    for s in range(30):
        base = 120 + rng.normal(0, 15); tr = -0.8 + rng.normal(0, 0.2)
        for yr in range(1970, 2001):
            eff = -20 * (yr - 1988) / 12 if (s == 0 and yr >= 1989) else 0   # 합성 처치효과
            rows.append((f"S{s}", yr, base + tr * (yr - 1970) + eff + rng.normal(0, 2), int(s == 0 and yr >= 1989)))
    df = pd.DataFrame(rows, columns=["State", "Year", "PacksPerCapita", "treated"])
    DATA_KIND = "SYNTHETIC (NOT real data)"
print("DATA_KIND:", DATA_KIND, "| shape:", df.shape, "| 단위 수:", df["State"].nunique())
wide = df.pivot(index="Year", columns="State", values="PacksPerCapita")
print("패널 균형 여부(결측 수):", int(wide.isna().sum().sum()))
wide.iloc[:, :8].plot(legend=False, title=DATA_KIND); plt.show()

## 점검 체크리스트
- 처치 표시(`treated`)가 처치 시점 이후 모든 기간에서 1인가, 처치 전 기간의 사전추세는 평행한가
- 도너풀 오염(유사 정책을 시행한 주)을 제외했는가
- 정책 시점의 정의·발표 시점 vs 시행 시점
- 국내 자료(KOSIS, data.go.kr)는 키 발급 후 별도 셀로 추가. 통계표 ID는 공식 포털에서 확인